**SML PROJECT**

# Problem Formulation & Synthetic Dataset Generation

### Operational Rationale & Methodology
As we all know that, campus dining halls experience severe daily demand volatility caused by conflicting calendar rhythms, weather shifts, exam schedules, and financial allowance cycles. To build a robust prediction pipeline prior to administrative data integration, we synthesized a primary 730-day (2-year) dataset grounded in real-world campus operational logic.The reason we synthesized the data is because generally dining halls does not keep track of thier meals served. So we are trying to mimic the real dataset. Every daily observation combines baseline demand with domain-specific additive effects and stochastic noise:

$$\text{Meals\_Served} = \text{Base} + \text{Weather_Impact} + \text{Menu_Impact} + \text{Month_End_Impact} + \text{Exam_Impact} - \text{Weekend_Drop} + \text{Vacation_Drop} + \epsilon$$

Where $\epsilon \sim \mathcal{N}(0, 22^2)$ ensures realistic statistical variation. This flow help us to generate the data that accepts real canteen log data without structural code changes.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer

# Set random seed
np.random.seed(42)

def generate_canteen_demand_data():

    # 1. TIMELINE GENERATION (2 Full Academic Years = 730 Days)

    dates = pd.date_range(start='2024-01-01', periods=730, freq='D')
    df = pd.DataFrame({'Date': dates})

    # Extract date attributes
    df['Day_of_Week'] = df['Date'].dt.day_name()
    df['Day_Num'] = df['Date'].dt.dayofweek  # 0=Monday, 6=Sunday
    df['Day_of_Month'] = df['Date'].dt.day
    df['Month'] = df['Date'].dt.month



    # A. Exam Weeks (4 periods/year: March, May, Oct, Dec mid/final exams)
    df['Is_Exam_Week'] = 0
    exam_mask = (df['Month'].isin([3, 5, 10, 12])) & (df['Day_of_Month'] >= 10) & (df['Day_of_Month'] <= 21)
    df.loc[exam_mask, 'Is_Exam_Week'] = 1

    # B. Vacations & Major Breaks (Summer: Jun-Jul, Winter: Late Dec-Early Jan)
    df['Is_Vacation'] = 0
    vacation_mask = (df['Month'].isin([6, 7])) | ((df['Month'] == 12) & (df['Day_of_Month'] >= 22)) | ((df['Month'] == 1) & (df['Day_of_Month'] <= 10))
    df.loc[vacation_mask, 'Is_Vacation'] = 1

    # C. Temperature (°C) with seasonal variation (10°C to 40°C range)
    seasonal_temp = 25 + 10 * np.sin(2 * np.pi * (df['Date'].dt.dayofyear - 80) / 365)
    df['Temperature'] = np.round(seasonal_temp + np.random.normal(0, 2.5, len(df)), 1)

    # D. Rain Precipitation (mm) (Higher in monsoon months Jun-Sep)
    is_monsoon = df['Month'].isin([6, 7, 8, 9])
    rain_prob = np.where(is_monsoon, 0.38, 0.12)
    rain_occurred = np.random.binomial(1, rain_prob)
    df['Rain_Precipitation_mm'] = np.round(rain_occurred * np.random.exponential(scale=12.0, size=len(df)), 1)

    # E. Special Menu Flag (Biryani/Paneer specials ~2 times/week on active days)
    menu_prob = np.where(df['Is_Vacation'] == 1, 0.05, 0.25)
    df['Special_Menu_Flag'] = np.random.binomial(1, menu_prob)

    # F. End of Month Effect (25th - 31st: Allowance depletion spikes canteen visits)
    df['Is_Month_End'] = (df['Day_of_Month'] >= 25).astype(int)

    # G. Hostel Occupancy Rate (%)
    # Term days ~88% base, vacations ~20% base, drops on weekends (Sat/Sun exodus)
    base_occ = np.where(df['Is_Vacation'] == 1, 20.0, 88.0)
    weekend_occ_drop = np.where(df['Day_Num'].isin([5, 6]), 25.0, 0.0)
    df['Hostel_Occupancy_Rate'] = np.clip(
        np.round(base_occ - weekend_occ_drop + np.random.normal(0, 4.0, len(df)), 1),
        10.0, 100.0
    )


    # Formula: Meals_Served = Base + Weather_Impact + Menu_Impact
    #                         + Month_End_Impact - Weekend_Drop + Random_Noise

    base = 500.0

    # Weather Impact (+20% to +35% demand spike on rainy days; slight drop in severe heat)
    rain_impact = np.where(df['Rain_Precipitation_mm'] > 0,
                           np.minimum(80.0 + df['Rain_Precipitation_mm'] * 3.5, 175.0),
                           0.0)
    temp_impact = np.where(df['Temperature'] > 35.0, -25.0, 0.0)
    weather_impact = rain_impact + temp_impact

    # Menu Impact (+110 meals on special menu days)
    menu_impact = df['Special_Menu_Flag'] * 110.0

    # Month End Impact (+10% to +15% boost, ~65 meals)
    month_end_impact = df['Is_Month_End'] * 65.0

    # Exam Week Impact (+85 meals due to students remaining on campus)
    exam_impact = df['Is_Exam_Week'] * 85.0

    # Weekend Drop (Mon-Thu baseline, Friday drop -40, Saturday -160, Sunday -200)
    dow_effects = {0: 10, 1: 35, 2: 40, 3: 35, 4: -40, 5: -160, 6: -200}
    weekend_drop = -df['Day_Num'].map(dow_effects)

    # Vacation Impact (-280 meals during breaks)
    vacation_impact = df['Is_Vacation'] * -280.0

    # Hostel Occupancy Scaling
    occupancy_impact = (df['Hostel_Occupancy_Rate'] - 75.0) * 2.5

    # Random Gaussian Noise (Std dev ~22 for realistic variation)
    random_noise = np.random.normal(0, 22.0, len(df))

    # Calculate total Meals Served
    raw_meals = (base
                 + weather_impact
                 + menu_impact
                 + month_end_impact
                 + exam_impact
                 - weekend_drop
                 + vacation_impact
                 + occupancy_impact
                 + random_noise)

    # Ensure Target Variable is a positive integer
    df['Meals_Served'] = np.maximum(30, np.round(raw_meals)).astype(int)


    output_columns = [
        'Date', 'Day_of_Week', 'Is_Exam_Week', 'Is_Vacation',
        'Temperature', 'Rain_Precipitation_mm', 'Hostel_Occupancy_Rate',
        'Special_Menu_Flag', 'Is_Month_End', 'Meals_Served'
    ]

    final_df = df[output_columns].copy()
    final_df.to_csv('canteen_demand.csv', index=False)

    print("✓ Dataset successfully generated and saved to 'canteen_demand.csv'!")
    return final_df


canteen_df = generate_canteen_demand_data()

print("\n--- First 5 Rows ---")
print(canteen_df.head())

print("\n--- Feature Summary Statistics ---")
print(canteen_df.describe().T[['mean', 'std', 'min', '50%', 'max']])

# Exploratory Data Analysis (EDA) & Preprocessing Pipeline



In [ ]:
# Set plot style
sns.set_theme(style="whitegrid")

# 1. LOAD DATASET
df = pd.read_csv('canteen_demand.csv', parse_dates=['Date'])

print("--- Dataset Shape ---")
print(f"Total Rows: {df.shape[0]}, Total Columns: {df.shape[1]}\n")

# 2. EXPLORATORY DATA ANALYSIS (EDA)
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# Plot A: Meal Demand Distribution
sns.histplot(df['Meals_Served'], kde=True, ax=axes[0, 0], color='skyblue')
axes[0, 0].set_title('Distribution of Daily Meals Served')
axes[0, 0].set_xlabel('Meals Served')

# Plot B: Day of Week Impact
order = ['Monday', 'Tuesday', 'Wednesday', 'Thursday', 'Friday', 'Saturday', 'Sunday']
sns.boxplot(data=df, x='Day_of_Week', y='Meals_Served', order=order, ax=axes[0, 1], palette='Blues_d')
axes[0, 1].set_title('Meal Demand by Day of Week')
axes[0, 1].tick_params(axis='x', rotation=30)

# Plot C: Rain Impact on Demand
sns.scatterplot(data=df, x='Rain_Precipitation_mm', y='Meals_Served', hue='Special_Menu_Flag', ax=axes[1, 0], alpha=0.7)
axes[1, 0].set_title('Rainfall (mm) vs Meals Served')

# Plot D: Exam Week & Month End Effects
sns.barplot(data=df, x='Is_Exam_Week', y='Meals_Served', hue='Is_Month_End', ax=axes[1, 1], palette='viridis')
axes[1, 1].set_title('Exam Week vs Month-End Effect')
axes[1, 1].set_xticklabels(['Regular Week', 'Exam Week'])

plt.tight_layout()
plt.savefig('eda_plots.png', dpi=300)
plt.show()

# Print Feature Correlation Matrix with Target Variable
numerical_cols = df.select_dtypes(include=[np.number]).columns
correlation_matrix = df[numerical_cols].corr()
print("\n--- Correlation with Target (Meals_Served) ---")
print(correlation_matrix['Meals_Served'].sort_values(ascending=False))

# 3. FEATURE PREPROCESSING & ENCODING
# Features (X) and Target (y)
X = df.drop(columns=['Date', 'Meals_Served'])
y = df['Meals_Served']

# Identify column categories
categorical_features = ['Day_of_Week']
numerical_features = ['Temperature', 'Rain_Precipitation_mm', 'Hostel_Occupancy_Rate']
binary_features = ['Is_Exam_Week', 'Is_Vacation', 'Special_Menu_Flag', 'Is_Month_End']

# Define Column Transformer for scaling & encoding
preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), numerical_features),
        ('cat', OneHotEncoder(drop='first', sparse_output=False), categorical_features),
        ('bin', 'passthrough', binary_features)
    ]
)

# 4. CHRONOLOGICAL TRAIN-TEST SPLIT (80% Train, 20% Test)
split_idx = int(len(df) * 0.8)

X_train_raw = X.iloc[:split_idx]
X_test_raw = X.iloc[split_idx:]
y_train = y.iloc[:split_idx]
y_test = y.iloc[split_idx:]

# Fit transformer on training set and transform both train and test sets
X_train_scaled = preprocessor.fit_transform(X_train_raw)
X_test_scaled = preprocessor.transform(X_test_raw)

# Extract generated feature names
cat_encoder = preprocessor.named_transformers_['cat']
encoded_cat_names = list(cat_encoder.get_feature_names_out(categorical_features))
feature_names = numerical_features + encoded_cat_names + binary_features

# Convert back to DataFrame for clean inspection
X_train = pd.DataFrame(X_train_scaled, columns=feature_names)
X_test = pd.DataFrame(X_test_scaled, columns=feature_names)

print("\n✓ Preprocessing Complete!")
print(f"Training set shape: {X_train.shape} (584 days)")
print(f"Test set shape: {X_test.shape} (146 days)")

### Data Inspection & Engineering
Before model training, we perform exploratory visual analysis and rigorous feature preprocessing to transform raw attributes into machine-readable matrices:
1. **Domain Hypothesis Validation:** Visualizing the non-linear impact of rainfall spikes (>10mm) and checking strong correlations like Hostel_Occupancy_Rate ($r = 0.919$).
2. **Categorical Encoding:** Applying OneHotEncoder(drop='first') to Day_of_Week to eliminate implicit numerical ordering.
3. **Feature Scaling:** Standardizing continuous numerical features ($\mu=0, \sigma=1$) using StandardScaler to prevent feature dominance in regularized regression.
4. **Chronological Splitting:** Enforcing a strict 80/20 time-series split (584 train days / 146 test days) to evaluate models on future unseen data without temporal data leakage.

Statistical Regression Modeling & Performance Evaluation

### Model Training & Comparative Benchmark
We implement and evaluate four distinct regression architectures across statistical performance metrics (MAE, RMSE, and $R^2$):
* **Linear Regression (OLS):** Serves as our baseline statistical model.
* **Ridge ($L_2$) & Lasso ($L_1$) Regression:** Penalize large parameters to mitigate collinearity between occupancy and vacation indicators.
* **Random Forest Regressor:** An ensemble tree algorithm designed to capture complex, non-linear feature interactions (such as combined rainy exam days).



In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.linear_model import LinearRegression, Ridge, Lasso
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

# 1. INITIALIZE MODELS
models = {
    "Linear Regression (OLS)": LinearRegression(),
    "Ridge Regression (L2)": Ridge(alpha=10.0),
    "Lasso Regression (L1)": Lasso(alpha=1.0),
    "Random Forest Regressor": RandomForestRegressor(n_estimators=100, random_state=42, max_depth=10)
}

# 2. TRAIN & EVALUATE
results = []

for name, model in models.items():
    # Fit model on scaled training data
    model.fit(X_train, y_train)

    # Predict on unseen test data
    y_pred = model.predict(X_test)

    # Calculate performance metrics
    mae = mean_absolute_error(y_test, y_pred)
    rmse = np.sqrt(mean_squared_error(y_test, y_pred))
    r2 = r2_score(y_test, y_pred)

    results.append({
        "Model": name,
        "MAE (Meals)": round(mae, 2),
        "RMSE (Meals)": round(rmse, 2),
        "R² Score": round(r2, 4)
    })

# Convert performance table to DataFrame
results_df = pd.DataFrame(results).sort_values(by="R² Score", ascending=False)

print("--- Model Performance Comparison (Test Set) ---")
print(results_df.to_string(index=False))

# 3. FEATURE IMPORTANCE (Random Forest)
rf_model = models["Random Forest Regressor"]
importances = pd.Series(rf_model.feature_importances_, index=X_train.columns).sort_values(ascending=True)

plt.figure(figsize=(10, 6))
importances.plot(kind='barh', color='teal')
plt.title('Feature Importance Analysis (Random Forest Regressor)')
plt.xlabel('Relative Importance Weight')
plt.tight_layout()
plt.savefig('feature_importance.png', dpi=300)
plt.show()

# 4. PREDICTION VS ACTUAL PLOT FOR BEST MODEL
best_pred = rf_model.predict(X_test)

plt.figure(figsize=(12, 5))
plt.plot(y_test.values[:60], label='Actual Meals Served', marker='o', color='black', alpha=0.7)
plt.plot(best_pred[:60], label='Predicted Meals (Random Forest)', marker='s', color='crimson', linestyle='--')
plt.title('60-Day Test Snapshot: Actual vs Predicted Canteen Meal Demand')
plt.xlabel('Test Days')
plt.ylabel('Meals Served')
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('actual_vs_predicted.png', dpi=300)
plt.show()

### Key Results Summary
* **Top Performing Model:** Random Forest Regressor ($R^2 = 0.9570$, MAE = 26.65 meals).
* **Operational Accuracy:** The model achieves a relative prediction error under **6%** across daily average meal volumes (~440 meals), successfully capturing sharp demand surges and weekend drop-offs.

#Further Improvement
So we are continuosly trying to tune the used parameter for better training and getting overall better results.